## Tools

### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running the code . Tools are pairings of:
#####    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
#####    2. A function or coroutine to execute


In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
# response = model.invoke("what is AI?")
# response

In [49]:
from langchain.tools import tool

@tool 
def get_weather(location:str)->str:
    """Get the weather of a location"""
    return f"It's cloudy in {location}"

@tool
def get_userinfo():
    """ Get the user info like who i am"""
    return "you are Alok , Alok is a software developer with 3 years of experience in development"

tools_map = {
    "get_weather": get_weather,
    "get_userinfo": get_userinfo
}
model_with_tools=model.bind_tools([get_weather,get_userinfo])

In [37]:
# response = model_with_tools.invoke("What's the weather at Bhubaneswar?")
response = model_with_tools.invoke("Who i am ? and What's the weather at Bhubaneswar?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: The user asks two questions: "Who i am ?" and "What\'s the weather at Bhubaneswar?"\n2.  **Identify Required Tools**:\n   - "Who i am ?" -> `get_userinfo` tool\n   - "What\'s the weather at Bhubaneswar?" -> `get_weather` tool with location="Bhubaneswar"\n3.  **Prepare Tool Calls**:\n   - `get_userinfo` (no parameters needed)\n   - `get_weather` with `location` = "Bhubaneswar"\n4.  **Execute Tool Calls**: Call both tools.\n5.  **Process Responses**: Combine the results to answer both questions clearly.✅\n   - `get_userinfo` will return user details.\n   - `get_weather` will return weather info for Bhubaneswar.\n   - Format the final response accordingly.🌟\n   - Note: I will make the calls now. \n   - Wait, I should generate the tool calls first.🔧\n   - `get_userinfo` -> {}\n   - `get_weather` -> {"location": "Bhubaneswar"}\n   - Proceed. \n   - *Self-Correction/Verification*: The tool defi

### Tool Execution Loops

In [ ]:
#step-1 : Model generates tool calls
messages = [{"role":"user","content":"What's the weather at Bhubaneswar?"}]
# messages = [{"role":"user","content":"Hi how are you"}]
# messages = [{"role":"user","content":"Who i am ? and What's the weather at Bhubaneswar?"}]

while True:
    # 1. Ask the model
    response = model_with_tools.invoke(messages)
    # Append the model's response to the conversation history
    messages.append(response)

     # 2. If the model didn't request any tool calls, we are done!
    if not response.tool_calls:
        print("\n=== Final Answer ===")
        print(response.content)
        break

    #3. If there are tool calls , execute them
    # print(f"\n--- Model requested {len(response.tool_calls)} tool(s) ---")
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        print(f"Executing: {tool_name} with args {tool_call["args"]}")
        tool_result = tools_map[tool_name].invoke(tool_call)
        # Append the tool's result to the message list
        messages.append(tool_result)
    



--- Model requested 1 tool(s) ---

=== Final Answer ===
It's currently cloudy in Bhubaneswar.
